In [1]:
# import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

In [2]:
dataset = pd.read_csv('./Melbourne_housing_FULL.csv')
print(dataset.columns)
dataset.head(5)

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'],
      dtype='str')


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,68 Studley St,2,h,NaN,SS,Jellis,3/09/2016,2.5,3067.0,...,1.0,1.0,126.0,NaN,NaN,Yarra City Council,-37.8014,144.9958,Northern Metropolitan,4019.0
1,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra City Council,-37.7996,144.9984,Northern Metropolitan,4019.0
2,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra City Council,-37.8079,144.9934,Northern Metropolitan,4019.0
3,Abbotsford,18/659 Victoria St,3,u,NaN,VB,Rounds,4/02/2016,2.5,3067.0,...,2.0,1.0,0.0,NaN,NaN,Yarra City Council,-37.8114,145.0116,Northern Metropolitan,4019.0
4,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra City Council,-37.8093,144.9944,Northern Metropolitan,4019.0


In [13]:
# Predicts prices from other features
feature_columns = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Distance', 'Bedroom2', 'Bathroom', 'Landsize', 'YearBuilt', 'CouncilArea', 'Regionname', 'Propertycount']
target_column = 'Price'
df_cropped = dataset[feature_columns+[target_column]]
df_cropped.head()

,Suburb,Rooms,Type,Method,SellerG,Distance,Bedroom2,Bathroom,Landsize,YearBuilt,CouncilArea,Regionname,Propertycount,Price
0,Abbotsford,2,h,SS,Jellis,2.5,2.0,1.0,126.0,NaN,Yarra City Council,Northern Metropolitan,4019.0,NaN
1,Abbotsford,2,h,S,Biggin,2.5,2.0,1.0,202.0,NaN,Yarra City Council,Northern Metropolitan,4019.0,1480000.0
2,Abbotsford,2,h,S,Biggin,2.5,2.0,1.0,156.0,1900.0,Yarra City Council,Northern Metropolitan,4019.0,1035000.0
3,Abbotsford,3,u,VB,Rounds,2.5,3.0,2.0,0.0,NaN,Yarra City Council,Northern Metropolitan,4019.0,NaN
4,Abbotsford,3,h,SP,Biggin,2.5,3.0,2.0,134.0,1900.0,Yarra City Council,Northern Metropolitan,4019.0,1465000.0


In [14]:
print(df_cropped.shape)
df_cropped.isnull().sum()

(34857, 14)


Suburb               0
Rooms                0
Type                 0
Method               0
SellerG              0
Distance             1
Bedroom2          8217
Bathroom          8226
Landsize         11810
YearBuilt        19306
CouncilArea          3
Regionname           3
Propertycount        3
Price             7610
dtype: int64

In [17]:
# Null handling
df_cropped = df_cropped.dropna(axis=0, subset=['Distance', 'Price', 'CouncilArea', 'Regionname', 'Propertycount'])
df = df_cropped.fillna({
    'Bedroom2': 0,
    'Bathroom': 0,
    'Landsize': df_cropped['Landsize'].mean(),
    'YearBuilt': df_cropped['YearBuilt'].mode()[0],
})
df.isnull().sum()

Suburb           0
Rooms            0
Type             0
Method           0
SellerG          0
Distance         0
Bedroom2         0
Bathroom         0
Landsize         0
YearBuilt        0
CouncilArea      0
Regionname       0
Propertycount    0
Price            0
dtype: int64

In [20]:
categorical_cols = ['Suburb', 'Type', 'Method', 'SellerG', 'CouncilArea', 'Regionname']
df_encoded = pd.get_dummies(df, dtype=int)

In [21]:
X = df_encoded.drop(columns=[target_column])
y = df_encoded[target_column]

In [22]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
def compare_regressions(X, y):
    """"Adopts regularization methods to compare"""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    scaler = StandardScaler()
    ## Initial scaling
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    # L1 norm penalty
    lasso = Lasso(alpha=0.1)
    # L2 norm -enalty
    ridge = Ridge(alpha=1.0)
    lasso.fit(X_train_scaled, y_train)
    ridge.fit(X_train_scaled, y_train)
    y_pred_lasso = lasso.predict(X_test_scaled)
    y_pred_ridge = ridge.predict(X_test_scaled)
    print(f"Lasso Coefficient: {r2_score(y_pred_lasso, y_test)}")
    print(f"Ridge Coefficient: {r2_score(y_pred_ridge, y_test)}")
    print(f"Lasso socre: {lasso.score(X_test_scaled, y_test)} vs Ridge score: {ridge.score(X_test_scaled, y_test)}")
compare_regressions(X, y)

Lasso Coefficient: 0.5643104571094848
Ridge Coefficient: 0.5642311617632415
Lasso socre: 0.6947149846502874 vs Ridge score: 0.6949526400658842
